# Parent Atlas — `.okf` domain learning challenger (GRU vs lexical baseline)

**PROPOSAL_ONLY.** Run this in Google Colab or a separate Python environment. It does not crawl websites, call an LLM, modify your repo, or write to PostgreSQL/Qdrant/Valkey. It consumes **reviewed** `feature_label`/task examples and the existing checksummed NES domain palette. An included synthetic fixture is only a smoke-test demonstration; its metrics do not justify promotion.

**Sources to upload (optional):**

1. `atlas_reviewed_training_examples.jsonl` — one labeled record per line. Fields: `id`, `text`, `feature_label`, `domain`, `source_ref`, `source_group`, `source_revision`, `workspace_revision`, `label_status` = `REVIEWED`. Different `source_group` values should represent genuinely independent source groups; don't split revisions of one source across train and validation.
2. `nes-domain-palette-v1.json` — exact generated output from `scripts/atlas/data/` (not a second palette).
3. `.okf` keyword corpus YAML and its strict JSON schema — optionally inspect vocabulary; **do not** auto-label training examples from it.

Do **not** upload private legal evidence, passwords, credentials or proprietary code without reviewing Google Colab's external processing implications.

In [ ]:
from pathlib import Path
import os, json, re, random, hashlib, copy, math
import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report

SEED = 20261107
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Torch:', torch.__version__, '| Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0), 'free bytes:', torch.cuda.mem_get_info()[0])
OUTPUT = Path('/content/atlas_okf_shadow_outputs') if Path('/content').exists() else Path('atlas_okf_shadow_outputs')
OUTPUT.mkdir(parents=True, exist_ok=True)

## 1. Upload reviewed examples and the checksummed NES palette

Colab users can run the optional upload cell. With no reviewed input, the notebook uses the bundled **synthetic example fixture** and marks all outputs non-promotable. The strict `.okf/corpora/keyword-corpus.yaml` is reference material, **not** a set of reviewed domain labels.

In [ ]:
# Optional Colab interactive upload; uncomment only when needed.
# from google.colab import files
# files.upload()

TRAIN_PATH = Path('/content/atlas_reviewed_training_examples.jsonl')
if not TRAIN_PATH.exists(): TRAIN_PATH = Path('atlas_reviewed_training_examples.jsonl')
DEMO_PATH = Path('/content/example-labeled-tasks.jsonl')
if not DEMO_PATH.exists(): DEMO_PATH = Path('example-labeled-tasks.jsonl')
if not TRAIN_PATH.exists() and not DEMO_PATH.exists():
    # Synthetic in-notebook fallback. Not real Parent Atlas labels.
    descriptions = {
      'auth': ['check session permissions', 'verify case ownership', 'reject unauthorized login', 'guard user tokens', 'test access control', 'validate auth middleware'],
      'ui': ['render svelte rune', 'check button layout', 'fix keyboard focus', 'test component props', 'inspect modal styling', 'reconcile responsive toolbar'],
      'retrieval': ['rank query chunks', 'compare nearest neighbors', 'evaluate semantic search', 'test recall at ten', 'merge RRF candidates', 'trace hybrid search'],
      'network': ['check http timeout', 'retry grpc request', 'validate rpc payload', 'trace web socket', 'inspect api endpoint', 'verify tls session'],
      'database': ['review postgres join', 'test drizzle schema', 'verify transaction rollback', 'inspect pgvector index', 'reconcile sql migration', 'check database identity'],
      'cache': ['invalidate valkey key', 'measure cache hit', 'bound prefix warming', 'test expiry fallback', 'inspect LRU budget', 'verify centroid revision'],
      'agent': ['route mcp tool', 'compose ACE evidence', 'approve repair proposal', 'build tool selector', 'check agent execution', 'validate action receipt'],
      'graph': ['expand graph edges', 'calculate pagerank', 'analyze dependency graph', 'audit node provenance', 'trace concept joins', 'verify graph traversal'],
      'ml': ['test torch tensor', 'benchmark cuda gemm', 'train domain classifier', 'validate embedding model', 'measure quantization error', 'score cross encoder'],
      'general': ['update documentation note', 'review project overview', 'check broad guidance', 'write release summary', 'organize work notes', 'prepare team briefing'],
    }
    synthetic=[]
    for domain, texts in descriptions.items():
        for i, t in enumerate(texts):
            synthetic.append({'id':f'demo:{domain}:{i}', 'text':t, 'feature_label':f'{domain.upper()} demo',
              'domain':domain,'source_ref':f'demo/{domain}/{i}.md','source_group':f'demo:{domain}:{i}',
              'source_revision':'SYNTHETIC','workspace_revision':'SYNTHETIC','label_status':'EXAMPLE_ONLY'})
    DEMO_PATH.write_text(''.join(json.dumps(x)+'\n' for x in synthetic),encoding='utf-8')

using_demo = not TRAIN_PATH.exists()
path = DEMO_PATH if using_demo else TRAIN_PATH
rows=[json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]
required={'id','text','feature_label','domain','source_ref','source_group','source_revision','workspace_revision','label_status'}
assert rows, 'No rows'
assert all(required.issubset(r) for r in rows), 'Missing required fields'
assert len(set(r['id'] for r in rows))==len(rows), 'Duplicate IDs'
assert all(r['label_status']=='REVIEWED' for r in rows) if not using_demo else all(r['label_status']=='EXAMPLE_ONLY' for r in rows)
print('Rows:',len(rows),'mode:', 'SYNTHETIC_DEMO' if using_demo else 'REVIEWED_INPUT')

## 2. Verify palette authority; optionally validate `.okf` corpus

The generated palette uses **separate** `lutByte` (domain routing) and `nesHex` (display/color) fields. `0x00` means unresolved. The notebook refuses unknown domain names and does not train new palette bytes. If the exact palette file is absent, it still trains a classifier, but suppresses NES mapping in the output.

In [ ]:
PALETTE_PATH = Path('/content/nes-domain-palette-v1.json')
if not PALETTE_PATH.exists(): PALETTE_PATH = Path('nes-domain-palette-v1.json')
PALETTE = None
if PALETTE_PATH.is_file():
    PALETTE = json.loads(PALETTE_PATH.read_text(encoding='utf-8'))
    expect = PALETTE.get('checksum', '')
    body = {k:v for k,v in PALETTE.items() if k != 'checksum'}
    actual='sha256:'+hashlib.sha256(json.dumps(body,separators=(',',':'),ensure_ascii=False).encode()).hexdigest()
    assert actual == expect, f'NES palette checksum failed: {actual}'
    assert PALETTE['schema']=='atlas.nes-domain-palette.v1'
    assert PALETTE['unresolved']['lutByte']==0
    print('Palette:',PALETTE['revision'],'validated SHA:', actual)
else:
    print('NES palette not supplied — no byte/color mapping will be emitted')

allowed_domains = set(PALETTE['domains']) if PALETTE is not None else {
    'auth','ui','retrieval','network','database','cache','agent','graph','ml','general'
}
assert all(r['domain'] in allowed_domains for r in rows), 'Unrecognized domain labels'

# An optional strict corpus inspection. Do not assign supervised domain labels from topic ids.
corpus_path=Path('/content/keyword-corpus.yaml')
schema_path=Path('/content/keyword-corpus.schema.json')
if corpus_path.is_file() and schema_path.is_file():
    import yaml, jsonschema
    corpus=yaml.safe_load(corpus_path.read_text(encoding='utf-8'))
    jsonschema.validate(corpus,json.loads(schema_path.read_text(encoding='utf-8')))
    assert corpus['canonical_authority'] is False
    assert corpus['indexing']['searchable'] is False
    print('Reference corpus valid:',len(corpus['entries']),'entries; not used as labels')

## 3. Leakage-aware source-group split + deterministic CPU lexical baseline

A credible dataset needs independently reviewed labels, enough examples for each class, and independent source groups. We split by group within each domain so variants from the same document don't leak between train and validation. The TF-IDF classifier is the baseline to beat; a GRU is not automatically better.

In [ ]:
from collections import defaultdict
by_domain=defaultdict(lambda: defaultdict(list))
for r in rows: by_domain[r['domain']][r['source_group']].append(r)
assert all(len(groups)>=3 for groups in by_domain.values()), 'Need >=3 independent source groups per domain'
train_rows=[]; val_rows=[]
for domain, groups in sorted(by_domain.items()):
    names=sorted(groups)
    random.Random(SEED+sum(map(ord,domain))).shuffle(names)
    n_val=max(1,round(len(names)*.25))
    valnames=set(names[:n_val])
    for name,rs in groups.items(): (val_rows if name in valnames else train_rows).extend(rs)
assert {r['source_group'] for r in train_rows}.isdisjoint({r['source_group'] for r in val_rows})
labels=sorted(by_domain)
label_to_i={name:i for i,name in enumerate(labels)}
text_of=lambda r: f"{r['feature_label']} | {r['text']}"  # no source ids as model input
Xtr=[text_of(r) for r in train_rows]; Xva=[text_of(r) for r in val_rows]
ytr=[label_to_i[r['domain']] for r in train_rows]; yva=[label_to_i[r['domain']] for r in val_rows]
vectorizer=TfidfVectorizer(analyzer='char',ngram_range=(2,4),max_features=12000)
Xtrain=vectorizer.fit_transform(Xtr); Xvalid=vectorizer.transform(Xva)
baseline=LogisticRegression(max_iter=500,random_state=SEED)
baseline.fit(Xtrain,ytr)
baseline_preds=baseline.predict(Xvalid)
base_f1=f1_score(yva,baseline_preds,average='macro',zero_division=0)
print('Train',len(train_rows),'validation',len(val_rows),'labels',labels)
print('TF-IDF baseline accuracy:',round(accuracy_score(yva,baseline_preds),4),'macro-F1:',round(base_f1,4))

## 4. Byte-level PyTorch GRU challenger

Every UTF-8 byte becomes a token (1–256); 0 is padding. Truncate to 256 bytes per example (measure truncation). This is a **classification model**, not an EmbeddingGemma vector substitute and not a TurboVec codec.

In [ ]:
MAX_BYTES=256

def encode_texts(texts):
    encoded=[]; lengths=[]; truncated=0
    for t in texts:
        bs=t.encode('utf-8')
        truncated+=int(len(bs)>MAX_BYTES)
        tokens=[b+1 for b in bs[:MAX_BYTES]]
        lengths.append(max(1,len(tokens)))
        encoded.append((tokens+[0]*MAX_BYTES)[:MAX_BYTES])
    return torch.tensor(encoded,dtype=torch.long),torch.tensor(lengths,dtype=torch.long),truncated

x_train,len_train,trunc_train=encode_texts(Xtr)
x_val,len_val,trunc_val=encode_texts(Xva)
print('Truncation:',trunc_train,'train,',trunc_val,'validation')

class ByteGRU(nn.Module):
    def __init__(self,num_classes):
        super().__init__()
        self.embed=nn.Embedding(257,32,padding_idx=0)
        self.gru=nn.GRU(32,64,batch_first=True)
        self.head=nn.Linear(64,num_classes)
    def forward(self,x,lengths):
        e=self.embed(x)
        packed=nn.utils.rnn.pack_padded_sequence(e,lengths.cpu(),batch_first=True,enforce_sorted=False)
        _,h=self.gru(packed)
        return self.head(h[-1])

model=ByteGRU(len(labels)).to(DEVICE)
optimizer=torch.optim.AdamW(model.parameters(),lr=0.003,weight_decay=0.01)
loss_fn=nn.CrossEntropyLoss()
dataloader=DataLoader(TensorDataset(x_train,len_train,torch.tensor(ytr)),batch_size=16,shuffle=True)
best_f1=-1.; best_state=None; best_epoch=None; bad=0
for epoch in range(1,21):
    model.train()
    for xb,lb,yb in dataloader:
        xb,lb,yb=xb.to(DEVICE),lb.to(DEVICE),yb.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss=loss_fn(model(xb,lb),yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(),max_norm=1.0)
        optimizer.step()
    model.eval()
    with torch.inference_mode():
        logits=model(x_val.to(DEVICE),len_val.to(DEVICE))
        prob=torch.softmax(logits.float(),dim=-1).cpu()
        preds=prob.argmax(-1).tolist()
    f1=f1_score(yva,preds,average='macro',zero_division=0)
    if f1>best_f1+1e-6:
        best_f1=f1; best_epoch=epoch; best_state=copy.deepcopy(model.state_dict());bad=0
    else: bad+=1
    print(f'epoch={epoch:02d} validation_macro_f1={f1:.4f}')
    if bad>=5: break
model.load_state_dict(best_state)
print('GRU best epoch:',best_epoch,'macro F1:',round(best_f1,4),'baseline macro F1:',round(base_f1,4))

## 5. Export **proposal-only** metrics and evidence-linked predictions

A confidence threshold is a configurable proposal criterion, **not** proof of probability calibration. Production needs held-out calibration, comparison to the baseline, and a frozen judged task set. The model cannot synthesize a new `feature_label`, `task_id`, `lutByte` or `nesHex` authority.

In [ ]:
model.eval()
with torch.inference_mode():
    p=torch.softmax(model(x_val.to(DEVICE),len_val.to(DEVICE)).float(),dim=-1).cpu().numpy()
state_path=OUTPUT/'byte_gru_proposal_state.pt'
torch.save(model.cpu().state_dict(),state_path)
state_sha=hashlib.sha256(state_path.read_bytes()).hexdigest()
palette_sha=PALETTE.get('checksum') if PALETTE else None
input_sha=hashlib.sha256(path.read_bytes()).hexdigest()
threshold=0.70
proposals=[]
for row,pr in zip(val_rows,p):
    idx=int(np.argmax(pr)); confidence=float(pr[idx]); pred=labels[idx]
    admitted_label=pred if confidence>=threshold else 'UNRESOLVED'
    nes_entry=(PALETTE['domains'].get(pred) if PALETTE is not None and admitted_label!='UNRESOLVED' else None)
    proposals.append({
      'schema':'atlas.okf.domain-proposal.v1','status':'PROPOSAL_ONLY',
      'id':row['id'],'feature_label':row['feature_label'],
      'source_ref':row['source_ref'],'source_revision':row['source_revision'],
      'workspace_revision':row['workspace_revision'],
      'predicted_domain':admitted_label,'raw_predicted_domain':pred,
      'confidence_uncalibrated':round(confidence,6),
      'lutByte':nes_entry['lutByte'] if nes_entry else (0 if PALETTE else None),
      'nesHex':nes_entry['nesHex'] if nes_entry else None,
      'palette_revision':PALETTE['revision'] if PALETTE else None,
      'model_sha256':state_sha,'canonical_authority':False
    })
metrics={
    'schema':'atlas.okf.learning-shadow-receipt.v1',
    'status':'DEMO_ONLY' if using_demo else 'HELD_OUT_PROPOSAL_ONLY',
    'train_count':len(train_rows),'validation_count':len(val_rows),
    'labels':labels,'seed':SEED,'input_sha256':input_sha,'palette_checksum':palette_sha,
    'model_sha256':state_sha,'best_epoch':best_epoch,
    'tfidf_logistic_macro_f1':base_f1,'gru_macro_f1':best_f1,
    'validation_groups_disjoint':True,'truncation_train':trunc_train,'truncation_validation':trunc_val,
    'promotion_allowed':False,'pgvector_writes':0,'qdrant_writes':0,'valkey_writes':0
}
(OUTPUT/'learning-shadow-receipt.json').write_text(json.dumps(metrics,indent=2)+'\n')
(OUTPUT/'domain-proposals.jsonl').write_text(''.join(json.dumps(p)+'\n' for p in proposals))
print(json.dumps(metrics,indent=2))
print('Saved:',sorted(p.name for p in OUTPUT.iterdir()))

## 6. What to evaluate next (not performed here)

- Acquire documentation with **the existing** `python/atlas_okf_docs_pipeline.py` under an explicit, schema-validated source manifest. Avoid duplicating its crawler.
- Build reviewed task→domain examples with `feature_id`, `feature_label`, source/revision and independent labels. Source-group split is mandatory to avoid leakage.
- Compare deterministic term/AST baseline, TF-IDF, the GRU, and EmbeddingGemma 2 frozen embeddings on **exactly the same** held-out set. Do not reuse synthetic hash-seeded vectors as model embeddings.
- Accept no regression in retrieval Recall@10 or unsupported NES palette mapping. Train on Colab only with data you are allowed to send to third-party cloud infrastructure.
- QLoRA and CUDA Tile are separate challengers. Neither is required to generate a meaningful domain label or validate a binary codec.